In [27]:
# 0: Lib
import numpy as np
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
from pathlib import Path
from torchvision import transforms
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
from tqdm import tqdm
import time
import os
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from torch.utils.data import DataLoader, Subset, random_split



In [13]:
# # 2: Dataset Class: transfer CSV in PyTorch.
# class PersonalGazeDataset(Dataset):

#     def __init__(
#         self,
#         root_dir,
#         transform=None,
#         dataset_name="norm_labels.csv",
#         dataset_size=None,
#         read_all4once=True
#     ):

#         self.root_dir = Path(root_dir)
#         self.transform = transform
#         self.read_all4once = read_all4once

#         # CSV file
#         csv_file = self.root_dir / dataset_name
#         self.df = pd.read_csv(csv_file)

#         # Dataset size
#         self.dataset_size = (
#             dataset_size
#             if dataset_size is not None
#             else len(self.df)
#         )

#         # Check dataset size
#         if self.dataset_size > len(self.df):
#             raise ValueError(
#                 f"dataset_size ({self.dataset_size}) is larger "
#                 f"than the number of samples in the CSV "
#                 f"({len(self.df)})."
#             )

#         # Read all images into memory
#         if self.read_all4once:

#             # Create a dummy image to determine the transformed shape
#             img = Image.new("RGB", (500, 300))

#             if self.transform:
#                 out = self.transform(img)
#             else:
#                 out = transforms.ToTensor()(img)

#             self.images = torch.zeros(
#                 [self.dataset_size] + list(out.shape)
#             )

#             self.targets = torch.zeros(
#                 self.dataset_size,
#                 2
#             )

#         # Load dataset
#         for idx in tqdm(
#             range(self.dataset_size),
#             desc="Loading dataset"
#         ):

#             row = self.df.iloc[idx]

#             # Image path
#             image_path = self.root_dir / row["image_name"]

#             image = Image.open(
#                 image_path
#             ).convert("RGB")

#             # Target coordinates
#             self.targets[idx] = torch.tensor(
#                 [row["x"], row["y"]],
#                 dtype=torch.float32
#             )

#             # Image transformation
#             if self.transform:

#                 self.images[idx] = self.transform(image)

#             else:

#                 self.images[idx] = transforms.ToTensor()(image)

                

#     def __len__(self):

#         return self.dataset_size

        

#     def __getitem__(self, idx):

#         return self.images[idx], self.targets[idx]


In [29]:
# 1: calss PersonalGazeDataset
class PersonalGazeDataset(Dataset):

    def __init__(
        self,
        root_dir,
        transform=None,
        dataset_size=None,
        read_all4once=True
    ):

        self.root_dir = Path(root_dir)
        self.transform = transform
        self.read_all4once = read_all4once

        self.df = pd.read_csv(
            self.root_dir / "norm_labels.csv"
            # self.root_dir 
        )

        self.dataset_size = (
            dataset_size
            if dataset_size is not None
            else len(self.df)
        )


        if self.read_all4once:

            # Form des transformierten Bildes bestimmen
            img = Image.new("RGB", (500, 300))
            out = transform(img)

            self.images = torch.zeros(
                [self.dataset_size] + list(out.shape)
            )

            self.targets = torch.zeros(
                self.dataset_size,
                2
            )


            for idx in tqdm(range(self.dataset_size)):

                row = self.df.iloc[idx]

                image = Image.open(
                    self.root_dir
                    / "images"
                    / row["frame"]
                ).convert("RGB")


                self.targets[idx] = torch.tensor(
                    [
                        row["x"],
                        row["y"]
                    ],
                    dtype=torch.float32
                )


                if self.transform:
                    self.images[idx] = self.transform(image)
                else:
                    self.images[idx] = image

    def __len__(self):

        return self.dataset_size
        

    def get_raw_item(self, idx):
    
        image = self.images[idx].clone()
        target = self.targets[idx].clone()
    
        return image, target


    # def __getitem__(self, idx):

    #     if self.read_all4once:

    #         return (
    #             self.images[idx],
    #             self.targets[idx]
    #         )

    #     row = self.df.iloc[idx]

    #     image = Image.open(
    #         self.root_dir
    #         / "images"
    #         / row["frame"]
    #     ).convert("RGB")

    #     target = torch.tensor(
    #         [
    #             row["x"],
    #             row["y"]
    #         ],
    #         dtype=torch.float32
    #     )

    #     if self.transform:
    #         image = self.transform(image)

    #     return image, target

    def __getitem__(self, idx):

        image, target = self.get_raw_item(idx)
    
        if self.transform:
            image = self.transform(image)
    
        return image, target

In [14]:
class TransformSubset(Dataset):

    def __init__(
        self,
        dataset,
        indices,
        transform=None,
    ):

        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        original_idx = self.indices[idx]

        image, target = self.dataset.get_raw_item(
            original_idx
        )

        if self.transform:
            image = self.transform(image)

        return image, target

In [15]:
# 2: func diagonal_error
def diagonal_errors(model, loader, device):

    # Evaluation-Modus
    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets_all = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)

    mae = mean_absolute_error(targets_all, predictions)

    rmse = np.sqrt(mean_squared_error(targets_all, predictions))

    diagonal_error_pct = np.round(100 * (rmse / np.sqrt(2)), 5)

    print(f"MAE : {mae:.4f} \t RMSE: {rmse:.4f} ")
    # print(f"RMSE: {rmse:.4f}")
    print(f"Diagonal-Error %: {diagonal_error_pct:.4f} %")

    return mae, rmse, diagonal_error_pct

In [16]:
def evaluate_model(model, loader, device):

    model.eval()

    targets_list = []
    preds_list = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images)
            targets_list.append(targets.cpu().numpy())
            preds_list.append(preds.cpu().numpy())

    targets = np.concatenate(targets_list, axis=0)
    predictions = np.concatenate(preds_list, axis=0)
    

    errors = np.sqrt(
        np.sum(
            (predictions-targets)**2,
            axis=1
        )
    )

    return targets, predictions, errors

In [17]:

def create_splits(
    full_dataset,
    train_size,
    validat_size,
    test_size,
    split_type="random",
    seed=42,
    original_fps=60,
    frame_step=5,
    reset_interval=5,
    exclusion_time=0.5,
    test_dataset=None
):

    dataset_size = len(full_dataset)


    # ============================================================
    # A: Random Split
    # ============================================================

    if split_type == "random":

        generator = torch.Generator().manual_seed(seed)

        train_dataset, validat_dataset, test_dataset = random_split(
            full_dataset,
            [train_size, validat_size, test_size],
            generator=generator
        )



    # ============================================================
    # B1: Sequential Split
    # ============================================================

    elif split_type == "sequential":

        train_indices = list(
            range(0, train_size)
        )

        validat_indices = list(
            range(
                train_size,
                train_size + validat_size
            )
        )

        test_indices = list(
            range(
                train_size + validat_size,
                dataset_size
            )
        )

        train_dataset = Subset(
            full_dataset,
            train_indices
        )

        validat_dataset = Subset(
            full_dataset,
            validat_indices
        )

        test_dataset = Subset(
            full_dataset,
            test_indices
        )



    # ============================================================
    # B2: Test first, then Train + Validation
    # ============================================================

    elif split_type == "test_first_random":

        generator = torch.Generator().manual_seed(seed)

        train_valid_size = train_size + validat_size

        train_valid_dataset, test_dataset = random_split(
            full_dataset,
            [train_valid_size, test_size],
            generator=generator
        )

        train_dataset, validat_dataset = random_split(
            train_valid_dataset,
            [train_size, validat_size],
            generator=generator
        )


    # ============================================================
    # C: Video 1 = Train + Validation
    #    Video 2 = Test
    # ============================================================

    elif split_type == "different_videos":

        if test_dataset is None:

            raise ValueError(
                "Für split_type='different_videos' "
                "muss test_dataset angegeben werden."
            )


        # --------------------------------------------------------
        # Train + Validation aus Video 1
        # --------------------------------------------------------

        train_valid_size = train_size + validat_size

        if train_valid_size > len(full_dataset):

            raise ValueError(
                "Train + Validation sind größer als "
                "das Train/Validation-Video."
            )


        generator = torch.Generator().manual_seed(seed)


        train_dataset, validat_dataset = random_split(
            full_dataset,
            [train_size, validat_size],
            generator=generator
        )


        # --------------------------------------------------------
        # Test = komplettes zweites Video
        # --------------------------------------------------------

        test_dataset = test_dataset


        print("\n" + "=" * 60)
        print("DIFFERENT-VIDEO SPLIT")
        print("=" * 60)

        print(
            f"Train/Validation Video: "
            f"{len(full_dataset)} Samples"
        )

        print(
            f"Test Video: "
            f"{len(test_dataset)} Samples"
        )

        print("\nSplit:")

        print(
            f"Train:       {len(train_dataset)}"
        )

        print(
            f"Validation:  {len(validat_dataset)}"
        )

        print(
            f"Test:        {len(test_dataset)}"
        )

        print("=" * 60)


    # ============================================================
    # D: Reset-aware Random Split
    # ============================================================

    elif split_type == "reset_aware":

        # --------------------------------------------------------
        # Tatsächliche Samplingrate nach Frame-Subsampling
        # --------------------------------------------------------

        effective_fps = original_fps / frame_step


        # --------------------------------------------------------
        # Anzahl gespeicherter Samples pro Reset-Intervall
        # --------------------------------------------------------

        reset_samples = int(
            round(reset_interval * effective_fps)
        )

        if reset_samples <= 0:
            raise ValueError(
                "reset_samples muss größer als 0 sein."
            )


        # --------------------------------------------------------
        # Anzahl gespeicherter Samples, die nach jedem Reset
        # ausgeschlossen werden
        # --------------------------------------------------------

        exclusion_samples = int(
            round(exclusion_time * effective_fps)
        )


        print("\n" + "=" * 60)
        print("RESET-AWARE SPLIT")
        print("=" * 60)

        print(f"Original FPS:          {original_fps}")
        print(f"Frame Step:            {frame_step}")
        print(f"Effektive FPS:         {effective_fps}")

        print(f"\nReset alle:            {reset_interval} s")
        print(f"Reset-Samples:         {reset_samples}")

        print(f"\nAusschlusszeit:        {exclusion_time} s")
        print(f"Ausschluss-Samples:    {exclusion_samples}")


        # --------------------------------------------------------
        # Gültige und ausgeschlossene Indizes bestimmen
        # --------------------------------------------------------

        valid_indices = []
        excluded_indices = []


        for idx in range(dataset_size):

            position_in_reset = idx % reset_samples


            # Erste 0.5 Sekunden nach jedem Reset
            if position_in_reset < exclusion_samples:

                excluded_indices.append(idx)

            else:

                valid_indices.append(idx)


        print(f"\nGesamte Samples:       {dataset_size}")
        print(f"Ausgeschlossen:        {len(excluded_indices)}")
        print(f"Verwendbar:            {len(valid_indices)}")


        # --------------------------------------------------------
        # Prüfen, ob genügend Daten vorhanden sind
        # --------------------------------------------------------

        required_size = (
            train_size
            + validat_size
            + test_size
        )


        if required_size > len(valid_indices):

            raise ValueError(
                "\nNicht genügend gültige Samples!\n"
                f"Benötigt:   {required_size}\n"
                f"Verfügbar: {len(valid_indices)}"
            )


        # --------------------------------------------------------
        # Gültige Daten zufällig mischen
        # --------------------------------------------------------

        generator = torch.Generator().manual_seed(seed)

        permutation = torch.randperm(
            len(valid_indices),
            generator=generator
        ).tolist()


        shuffled_indices = [
            valid_indices[i]
            for i in permutation
        ]


        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------

        train_indices = shuffled_indices[
            :train_size
        ]


        # --------------------------------------------------------
        # Validation
        # --------------------------------------------------------

        validat_indices = shuffled_indices[
            train_size:
            train_size + validat_size
        ]


        # --------------------------------------------------------
        # Test
        # --------------------------------------------------------

        test_indices = shuffled_indices[
            train_size + validat_size:
            train_size + validat_size + test_size
        ]


        # --------------------------------------------------------
        # Subsets
        # --------------------------------------------------------

        train_dataset = Subset(
            full_dataset,
            train_indices
        )

        validat_dataset = Subset(
            full_dataset,
            validat_indices
        )

        test_dataset = Subset(
            full_dataset,
            test_indices
        )


        print("\nSplit:")
        print(f"Train:                 {len(train_dataset)}")
        print(f"Validation:            {len(validat_dataset)}")
        print(f"Test:                  {len(test_dataset)}")

        print("=" * 60)


    else:

        raise ValueError(
            f"Unbekannter split_type: {split_type}"
        )


    return (
        train_dataset,
        validat_dataset,
        test_dataset
    )

In [18]:
# Transformationen:
transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    transforms.RandomGrayscale(p=0.05),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.5)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
# ================================================
# Train Transform:
# ================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    transforms.RandomGrayscale(p=0.05),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.5)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ================================================
# Test/Validation Transform:
# ================================================

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [31]:
# Hypoparameter:

name_dataset_type = ['norm_labels.csv', 'labels.csv']
dataset_size_type = [[10000, 2000], [1000, 200]]
dataset_type = ["norm_subject", "norm_random"]
batch_size_type = [32, 64, 128]
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
epochs_num = [1, 2, 10, 25, 500]



optimizer_name = "AdamW"

# ##########################################################################
# 1.type Dataset: 'norm_labels.csv' or 'labels.csv'
dataset_name = name_dataset_type[0]                       # norm_labels.csv
# dataset_name = name_dataset_type[1]                       # labels.csv
print(f"\n dataset_name: \t {dataset_name}")


# 2.dataset_sizt: (10000 & 2000) or (1000, 200)
# def_dataset_size = dataset_size_type[0]                   # [10000, 2000]
def_dataset_size = dataset_size_type[1]                   # [1000, 200]
print(f"\n def_dataset_size: train: {def_dataset_size[0]}, \t test: {def_dataset_size[1]}")


# 3.batch_size: '32', '64' or '128'
batch_Size = batch_size_type[0]                           # 32
# batch_Size = batch_size_type[1]                           # 64
# batch_Size = batch_size_type[2]                           # 128
print(f"\n batch_Size: \t {batch_Size}")


# 4.type of dataset-split: "norm_subject_independed" or "norm_random"
# load model:
dataset_type = ["norm_subject", "norm_random"]

# load model:
#============================================================================
#                 normalize_subject_indipended 
#============================================================================
def_dataset = dataset_type[0]                             # norm_subject
saved_model = "./models/best_models/best_ResNet_Sigmoid_norm_subject.path"


#============================================================================
#                 normalize_random
#============================================================================
# def_dataset = dataset_type[1]                             # norm_random
# saved_model = "./models/best_models/best_ResNet_Sigmoid_norm_random.path"

print(f"\n def_dataset: \t {def_dataset}")

# load model:



# 5.learning_rate: '1e-3', '1e-4', '1e-5' or '1e-6'
lr_type = [1e-3, 1e-4, 1e-5, 1e-6] 
# learning_rate = lr_type[0]                                # 1e-3
learning_rate = lr_type[1]                                # 1e-4
# learning_rate = lr_type[2]                                # 1e-5
# learning_rate = lr_type[3]                                # 1e-6
print(f"\n learning_rate: \t {learning_rate}")


# 6.number of epochs: 1, 2, 10, 25 or 500
epochs = epochs_num[-1]                                   # 500
print(f"\n epochs: \t {epochs}")




weight_Decay = 1e-5

active_func = None

patience = 3               # after 3 Epochen without Optimierung has to stop training
print(f"\n patience: \t {patience}")





# session = "03"

# best_model_name = "./models/ResNet_best_optim-model_norm_subject_1000-200.path"

# last best model: with Sigmoid
# saved_model = "./models/best_models/best_ResNet_Sigmoid.path"


# load model:
saved_model = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"

#============================================================================
#                 normalize_subject_indipended 
#============================================================================
# saved_model1 = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"

#============================================================================
#                 normalize_random
#============================================================================
# saved_model2 = f"./models/best_models/best_ResNet_Sigmoid_{def_dataset}.path"



 dataset_name: 	 norm_labels.csv

 def_dataset_size: train: 1000, 	 test: 200

 batch_Size: 	 32

 def_dataset: 	 norm_subject

 learning_rate: 	 0.0001

 epochs: 	 500

 patience: 	 3


In [32]:
# 9: ResNet18 (Pretrainiertes Modell):
# Load:
def reset_model(act=None):
    model = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    model_name = model.__class__.__name__
    
    # ---------------------------------------
    # Activation function for the last layer
    # ---------------------------------------
    if act == "Sigmoid":
        last_layer = nn.Sigmoid()

    elif act == "Gaussian":
        last_layer = GaussianActivation(sigma=1.0)

    elif act == "ReLU":
        last_layer = nn.ReLU()

    elif act == "None":
        last_layer = nn.Identity()

    else:
        raise ValueError(
            f"Unknown activation function: {act}"
        )

    # ---------------------------------------
    # Replace original ResNet FC
    # ---------------------------------------
    model.fc = nn.Sequential(
        nn.Linear(
            model.fc.in_features,
            512
        ),
        nn.ReLU(),

        nn.Linear(
            512,
            256
        ),
        nn.ReLU(),

        nn.Dropout(0.2),

        nn.Linear(
            256,
            128
        ),
        nn.ReLU(),

        nn.Linear(
            128,
            2
        ),

        # Last activation
        last_layer
    )

    return model, model_name

In [33]:
# 10: GPU or CPU
# check:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [22]:
norm_labels_files = []
root_folder = Path("./personalization/")


for folder in sorted(root_folder.iterdir()):

    if folder.is_dir():

        norm_file = folder / "norm_labels.csv"


        if not norm_file.exists():

            # print(f"Fehlt: {norm_file.resolve()}\n\n")
            continue

        else:

            norm_labels_files.append(norm_file.as_posix())


for n in norm_labels_files:
    print(f"norm_file: {n}")

print(f"\n Anzahl der norm_files: {len(norm_labels_files)}\n")


sessions = []

for i in range(len(norm_labels_files)):
    path = str(norm_labels_files[i])
    session = path.split("/")[1]
    # print(f"Session: {session}")
    sessions.append(session)

print(f"Sessions: {sessions}")

print(f"\n Anzahl der Sessions: {len(sessions)}")

norm_file: personalization/01/norm_labels.csv
norm_file: personalization/02/norm_labels.csv
norm_file: personalization/03/norm_labels.csv
norm_file: personalization/04/norm_labels.csv
norm_file: personalization/05/norm_labels.csv
norm_file: personalization/06/norm_labels.csv
norm_file: personalization/07/norm_labels.csv
norm_file: personalization/08/norm_labels.csv
norm_file: personalization/09/norm_labels.csv
norm_file: personalization/10/norm_labels.csv
norm_file: personalization/11/norm_labels.csv
norm_file: personalization/12/norm_labels.csv
norm_file: personalization/13/norm_labels.csv
norm_file: personalization/14/norm_labels.csv
norm_file: personalization/15/norm_labels.csv
norm_file: personalization/16/norm_labels.csv
norm_file: personalization/17/norm_labels.csv
norm_file: personalization/18/norm_labels.csv

 Anzahl der norm_files: 18

Sessions: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18']

 Anzahl der Sessions: 1

In [30]:

for s in sessions:

    session = s

    print(f"\nSession: {session}")

    dirs = f"personalization/{session}"
    
    csv_path = Path(dirs) / "norm_labels.csv"
    
    if not csv_path.exists():
        raise FileNotFoundError(f"Fehlt: {csv_path.resolve()}")
    
    print(f"csv_pfad: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    dataset_size = len(df)
    
    print(f"Gesamte Daten: {dataset_size}")
    
    
    # ============================================================
    # Full Dataset
    # ============================================================
    
    # full_dataset = PersonalGazeDataset(
    #     root_dir=dirs,
    #     transform=None,
    # )
    full_dataset = PersonalGazeDataset(
        root_dir=dirs,
        transform=transform,
        # dataset_name=dataset_name
    )
    
    dataset_size = len(full_dataset)
    
    print(f"Dataset Size: {dataset_size}")
    
    # ============================================================
    # Split Sizes
    # ============================================================
    
    train_size = int(dataset_size * 0.70)
    validat_size = int(dataset_size * 0.15)
    
    test_size = (
        dataset_size
        - train_size
        - validat_size
    )


    # ============================================================
    # Random Split Indices
    # ============================================================
    
    generator = torch.Generator().manual_seed(42)
    
    indices = torch.randperm(
        dataset_size,
        generator=generator
    ).tolist()
    
    train_indices = indices[:train_size]
    
    validat_indices = indices[
        train_size:train_size + validat_size
    ]
    
    test_indices = indices[
        train_size + validat_size:
    ]
    
    # ============================================================
    # Datasets with Correct Transforms
    # ============================================================
    
    train_dataset = TransformSubset(
        full_dataset,
        train_indices,
        transform=train_transform,
    )
    
    validat_dataset = TransformSubset(
        full_dataset,
        validat_indices,
        transform=eval_transform,
    )
    
    test_dataset = TransformSubset(
        full_dataset,
        test_indices,
        transform=eval_transform,
    )
    
    print(f"Train:      {len(train_dataset)}")
    print(f"Validation: {len(validat_dataset)}")
    print(f"Test:       {len(test_dataset)}")
    
    

    # ========================================================================
    # Data Loading Strategies:
    # ========================================================================

    # A1: Random
    split = "a_random"
    train_dataset, validat_dataset, test_dataset = create_splits(
        full_dataset,
        train_size,
        validat_size,
        test_size,
        split_type="random",
        seed=42
    )
    
    
    print(f"Train:      {len(train_dataset)}")
    print(f"Validation: {len(validat_dataset)}")
    print(f"Test:       {len(test_dataset)}")



    
    
    
    # Loader:
    train_loader = DataLoader(
        train_dataset,
        batch_size=train_size,
        shuffle=True,
        num_workers=8,
        persistent_workers=True
    )
    
    
    
    validat_loader = DataLoader(
        validat_dataset,
        batch_size=validat_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )
    
    
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=test_size,
        shuffle=False,
        num_workers=8,
        persistent_workers=True
    )



    # ============================================================
    # Training:
    # ============================================================
    
    # acts = ['Sigmoid', 'Gaussian', 'ReLU', 'None']
    act = "Sigmoid"
    
    trainable_layers = "FC only"
    
    
    print("\n" + "=" * 70)
    print(f"{def_dataset} EXPERIMENT: ")
    
    
    print(f"\tDataset_type:         {dataset_name}")
    print(f"\tDataset split:        {def_dataset}")
    print(f"\tTrain size:           {def_dataset_size[0]}")
    print(f"\tTest size:            {def_dataset_size[1]}")
    print(f"\tBatch size:           {batch_Size}")
    print(f"\tLearning rate:        {learning_rate}")
    print(f"\tActivation:           {act}")
    print(f"\tTrainable layers:     {trainable_layers}")
    print(f"\tMax_Epochs:           {epochs}")
    print(f"\tPatience:             {patience}")
    print("=" * 70)
    
    
    
    
    if globals().get("bas_const_err") is None:
        bas_const_err = 27.245
    
    if globals().get("bas_rand_err") is None:
        bas_rand_err = 38.961
    
    
    
    # --------------------------------------------------------
    # Start total training timer
    # --------------------------------------------------------
    
    train_start = time.perf_counter()
    
    
    
    epochs = epochs
    
    
    
    model_output = './models/best_models/personal/'
    model_dir= Path(model_output)
    
    if not os.path.exists(model_dir):
        model_dir.mkdir(
            parents=True,
            exist_ok=True
        )
    
    
    diagrams_output = './results/best_diagrams/personal'
    diagrams_dir= Path(diagrams_output)
    
    if not os.path.exists(diagrams_dir):
        diagrams_dir.mkdir(
            parents=True,
            exist_ok=True
        )
    
    
    
    # ============================================================
    # Store train and test error curves
    # ============================================================
    
    # train_errors = []
    # test_errors = []
    
    best_error = float("inf")
    
    patience_counter = 0                        # Number Epochen without Optimierung
    improvements = 0                            # Number Epochen with Optimierung
    best_epoch = 0
    
    diag_train_errors = []
    diag_validat_errors  = []
    
    act_time_epochs = {}
    
    
    
    criterion = nn.L1Loss()
    
    criterion_base = nn.L1Loss()
    
    # --------------------------------------------------------
    # Reset model
    # --------------------------------------------------------
    
    model, model_name = reset_model(act=act)

    checkpoint = torch.load(
        # checkpoint_path,
        saved_model,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(checkpoint)
    
    model.to(device)
    
    # --------------------------------------------------------
    # Freeze all parameters
    # --------------------------------------------------------
    
    for param in model.parameters():
        param.requires_grad = False
    
    
    # --------------------------------------------------------
    # Unfreeze ONLY FC
    # --------------------------------------------------------
    
    for param in model.layer4.parameters():
        param.requires_grad=False
    
    
    for param in model.fc.parameters():
        param.requires_grad = True
    
    
    
    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_Decay
    )
    
    
    
    
    
    print('\n ',"=#=" * 25)
    print(f"\t\t Session: {session}")
    print(' ',"=#=" * 25)
    
    
    # ============================================================
    # Initial evaluation before training
    # ============================================================
    print("\n\t Initial evaluation: \n")
    
    model.eval()
    
    print("\n Train_error:")
    train_mae, train_rmse, train_diag_pct = diagonal_errors(
        model,
        train_loader,
        device
    )
    
    train_diag_pct = np.round(float(train_diag_pct), 4)
    # diag_train_errors.append(train_diag_pct)
    
    
    
    
    print("\n Validat_error:")
    validat_mae, validat_rmse, validat_diag_pct = diagonal_errors(
        model,
        validat_loader,
        device
    )
    
    validat_diag_pct = np.round(float(validat_diag_pct), 4)
    # diag_validat_errors.append(validat_diag_pct)
    
    
    
    print(
        f"Epoch 0 | "
        f"Train error: {train_diag_pct:.4f}% | "
        f"Validat error: {validat_diag_pct:.4f}%"
    )
    
    
    
    # Initial best test error
    if validat_diag_pct < best_error:  
        best_error = validat_diag_pct
        best_epoch = 0
    
    
    # save the better Modell
    torch.save(
        model.state_dict(),
        f"{model_output}"
        f"best_{model_name}_{act}_{def_dataset}_{session}.path"
    )
    
    # ============================================================
    # Training
    # ============================================================
    
    print("\n\n\t Training: \n")
    
    for epoch in range(epochs):
    
        epoch_start = time.perf_counter()
    
        model.train()
    
        running_loss = 0.0
    
    
        loop = tqdm(
            train_loader,
            desc=f"Normalized | Epoch {epoch + 1}"
        )
        
    
        for images, targets in loop:
            
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            preds = model(images)
    
            ################################
            # Loss :
            ################################
            
            loss = criterion(
                preds,
                targets
            )
    
    
            
            ################################
    
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    
            loop.set_postfix(
                loss=loss.item()
            )
    
        
    
        # ========================================================
        # Evaluation after epoch
        # ========================================================
    
        model.eval()
    
    
        # --------------------------------------------------------
        # Train error
        # --------------------------------------------------------
        print("\n Train:")
        train_mae, train_rmse, train_diag_pct = diagonal_errors(
            model,
            train_loader,
            device
        )
    
        train_diag_pct = np.round(float(train_diag_pct), 4)
        diag_train_errors.append(train_diag_pct)
    
    
        # --------------------------------------------------------
        # Test error
        # --------------------------------------------------------
        print("\n Validat:")
        validat_mae, validat_rmse, validat_diag_pct = diagonal_errors(
            model,
            validat_loader,
            device
        )
    
        validat_diag_pct = np.round(float(validat_diag_pct), 4)
        diag_validat_errors.append(validat_diag_pct)
    
    
        # --------------------------------------------------
        # Early Stopping
        # --------------------------------------------------
    
        if validat_diag_pct < best_error:
    
            best_error = validat_diag_pct
    
            best_epoch = epoch + 1
    
            patience_counter = 0
            improvements += 1
    
            print(
                f"\nValidat-Diagonal-Error: "
                f"{validat_diag_pct:.4f}% | "
                f"Improvement: {improvements}"
            )
    
            # save the better Modell
            torch.save(
                model.state_dict(),
                f"{model_output}"
                f"best_{model_name}_{act}_{def_dataset}_{session}.path"
            )
        
    
        else:
    
            # without imporovement
            patience_counter += 1
    
            print(
                f"\nPatience: "
                f"{patience_counter}/{patience}"
            )
    
    
            if patience_counter >= patience:
    
                print(
                    f"\nEarly Stopping after "
                    f"{epoch + 1} epochs."
                )
                print(
                    f"Imporovments totally: {improvements}"
                )
    
                break
        
    
        # ========================================================
        # Epoch information
        # ========================================================
    
        epoch_end = time.perf_counter()
    
        epoch_time = epoch_end - epoch_start
    
    
        print(
            f"\n[{datetime.now().strftime('%H:%M:%S')}] "
            f"Epoch {epoch + 1}: | " 
            f"{epoch_time:.2f} s | "
            f"Loss: {running_loss / len(train_loader):.4f} | "
            f"Train-Error: {train_diag_pct:.4f}% | "
            f"Test-Error: {validat_diag_pct:.4f}%"
        )
    
        torch.save(
            model.state_dict(),
            f"{model_output}"
            f"last_{model_name}_{act}_{def_dataset}_{session}.path"
        )
    
    
    
    # ============================================================
    # Total running time
    # ============================================================
    
    train_end = time.perf_counter()
    
    elapsed_running_time = train_end - train_start
    
    elapsed_minutes = elapsed_running_time / 60
    
    print(f"\ntotll running-tiems (s): {elapsed_running_time:.2f} s")
    print(f"totll running-tiems (min): {elapsed_minutes:.2f} min\n")
    
    # ============================================================
    # Number of completed epochs
    # ============================================================
    
    epochs_completed = len(diag_validat_errors)
    
    
    # # ============================================================
    # # Store normalized-dataset results
    # # ============================================================
    
    results[session] = {
        "split_type" : def_dataset,
        "train_errors": diag_train_errors,
        "validat_errors": diag_validat_errors,
        "best_validat_error": best_error,
        "running_time_minutes": elapsed_minutes,
        "epochs": epochs_completed,
        "best_epoch": best_epoch,
        "improvements": improvements
    }
    
    
    # ========================================================
    # Print result for this learning rate
    # ========================================================
    
    print("\n" + "=" * 70)
    print(f"{def_dataset.upper()} {session.upper()} RESULTS: ")
    print("=" * 70)
    
    print(
        f"Best Validation Error:   {norm_results['best_validat_error']:.4f}%"
    )
    print(
        f"Running Time:      {norm_results['running_time_minutes']:.2f} min"
    )
    print(
        f"Epochs:            {norm_results['epochs']}"
    )
    print(
        f"Best Epoch:        {norm_results['best_epoch']}"
    )
    print(
        f"Improvements:      {norm_results['improvements']}"
    )
    print("=" * 70)
    
    
    # ---------- Evaluation ----------
    
    # norm_subject:
    # saved_model1 = "./models/best_models/best_ResNet_Sigmoid_norm_subject.path"
    # saved_model1 = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"
    
    # norm_random:
    # saved_model2 = "./models/best_models/best_ResNet_Sigmoid_norm_random.path"
    # saved_model2 = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"

    saved_model = f"{model_output}best_{model_name}_{act}_{def_dataset}_{session}.path"

    checkpoint = torch.load(
        saved_model,
        map_location=device
    )
    
    model.load_state_dict(checkpoint)
    
    model.to(device)
    
    model.eval()
    
    
    print(f"Test:\n")
    test_mae, test_rmse, test_diag_pct = diagonal_errors(model, test_loader, device)
    
    print(
            f"Final diagonal Test Error= {test_diag_pct:.4f}% "
        )
    test_diag_pct = np.round(float(test_diag_pct), 4)
    results[session]["test_errors"] = test_diag_pct





Session: 01
csv_pfad: personalization/01/norm_labels.csv
Gesamte Daten: 498


100%|█████████████████████████████████████████████████████████████████████████████████████████| 498/498 [00:04<00:00, 116.45it/s]


Dataset Size: 498
Train:      348
Validation: 74
Test:       76
Train:      348
Validation: 74
Test:       76

norm_subject EXPERIMENT: 
	Dataset_type:         norm_labels.csv
	Dataset split:        norm_subject
	Train size:           1000
	Test size:            200
	Batch size:           32
	Learning rate:        0.0001
	Activation:           Sigmoid
	Trainable layers:     FC only
	Max_Epochs:           500
	Patience:             3

  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=
		 Session: 01
  =#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#==#=

	 Initial evaluation: 


 Train_error:


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/ari/PycharmProjects/Ven/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ari/PycharmProjects/Ven/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ari/PycharmProjects/Ven/.venv/lib/python3.12/site-packages/torch/utils/data/dataset.py", line 443, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
            ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2219883/2212377604.py", line 117, in __getitem__
    image = self.transform(image)
            ^^^^^^^^^^^^^^^^^^^^^
  File "/home/ari/PycharmProjects/Ve